# Full Version (NOT Suitable for raining in Google Colab)

Link For the original: https://towardsdev.com/building-small-language-models-from-scratch-a-production-grade-engineering-guide-3a26fe9a73f7

In [1]:
!pip install -q torch
!pip install -q datasets
!pip install -q tiktoken

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np
import math

from datasets import load_dataset
import tiktoken

In [3]:
batch_size = 32
block_size = 128

n_embed = 384
n_head = 6
n_layer = 6

dropout = 0.1

learning_rate = 1e-4
max_iters = 20000

device = 'CUDA' if torch.cuda.is_available() else 'CPU'

print("Device: ", device)

Device:  CUDA


## Download Dataset

In [4]:
dataset = load_dataset("roneneldan/TinyStories")

train_ds = dataset["train"]
val_ds = dataset["validation"]

print(train_ds)
print(val_ds)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Dataset({
    features: ['text'],
    num_rows: 2119719
})
Dataset({
    features: ['text'],
    num_rows: 21990
})


## Tokenizer

In [5]:
enc = tiktoken.get_encoding("gpt2")

print(enc.encode("Hello world"))

[15496, 995]


## Tokenize Dataset

In [6]:
def process(example):
  ids = enc.encode_ordinary(example["text"])

  return {
      "ids": ids,
      "len": len(ids)
  }

tokenized = dataset.map(
    process,
    remove_columns=["text"],
    num_proc = 4
)

print(f"Tokenized dataset structure: {tokenized}")

Tokenized dataset structure: DatasetDict({
    train: Dataset({
        features: ['ids', 'len'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['ids', 'len'],
        num_rows: 21990
    })
})


## Create tain dataset and validation dataset

In [ ]:
for split in ["train", "validation"]:

  ids = []

  for item in tokenized[split]:
    ids.extend(item["ids"])

  arr = np.array(ids, dtype = np.uint16)

  arr.tofile(f"{split}.bin")

  print(
      split,
      arr.shape,
      arr.nbytes / 1024 / 1024,
      "MB"
  )


In [ ]:
def get_batch(split):

  data = np.memmap(
      "train.bin" if split=="train" else "validation.bin",
      dtype = np.uint16,
      mode="r"
  )

  ix = torch.randint(
      len(data)-block_size,
      (batch_size,)
  )

  x = torch.stack([
      torch.from_numpy(
          data[i:i+block_size].astype(np.int64)
      )

      for i in ix
  ])


  y = torch.stack([
      torch.from_numpy(
          data[i+1:i+1+block_size].astype(np.int64)
      )

      for i in ix
  ])


  return x.to(device), y.to(device)

## Attention Head

In [ ]:
class Head(nn.Module):

  def __init__(self, head_size):
    super().__init__()

    self.key = nn.Linear(n_embed, head_size, bias=False)
    self.query = nn.Linear(n_embed, head_size, bias=False)
    self.value = nn.Linear(n_embed, head_size, bias=False)

    self.dropout = nn.Dropout(dropout)

    self.register_buffer(
        "tril",
        torch.tril(torch.ones(block_size, block_size))
    )


  def forward(self,x):

    B, T, C = x.shape

    k = self.key(x)
    q = self.query(x)

    wei = q @ k.transpose(-2, -1)

    wei = wei * (k.shape[-1] ** -0.5)

    wei = wei.masked_fill(
        self.tril[:T, :T]==0,
        float("-inf")
    )

    wei = F.softmax(wei, dim=-1)

    wei = self.dropout(wei)

    v = self.value(x)

    out = wei @ v

    return out

## MultiHead Attention

In [ ]:
class MultiHeadAttention(nn.Module):

  def __init__(self, num_heads, head_size):
    super().__init__

    self.heads = nn.ModuleList([
        Head(head_size)
        for _ in range(num_heads)
    ])

    self.proj = nn.Linear(n_embed, n_embed)

    self.dropout = nn.Dropout(dropout)

  def forward(self, x):

    out = torch.cat(
        [h(x) for h in self.heads],
        dim = -1
    )

    out = self.dropout(
        self.proj(out)
    )

    return out

## FeedForward Network

In [ ]:
class FeedForward(nn.Module):

  def __init__(self):
    super().__init__()

    self.net = nn.Sequential(
        nn.Linear(n_embed, 4*n_embed),
        nn.GeLU(),
        nn.Linear(4*n_embed, n_embed),
        nn.Dropout(dropout)
    )

    def forward(self, x):
      return self.net(x)

## Transformer Block

In [ ]:
class Block(nn.Module):

  def __init__(self):
    super().__init__()

    head_size = n_embed // n_head

    self.sa = MultiHeadAttention(
        n_head,
        head_size,
    )

    self.ffwd = FeedForward()

    self.ln1 = nn.LayerNorm(n_embed)
    self.ln2 = nn.LayerNorm(n_embed)

  def forward(self, x):

    x = x + self.sa(self.ln1(x))

    x = x + self.ffwd(self.ln2(x))

    return x



## GPT Model

In [ ]:
class GPT(nn.Module):

  def __init__(self):
    super().__init__()

    self.token_embedding_table = nn.Embedding(
        50257,
        n_embed
    )

    self.position_embedding_table = nn.Embedding(
        block_size,
        n_embed
    )

    self.blocks = nn.Sequential(
        *[Block() for _ in range(n_layer)]
    )

    self.ln_f = nn.LayerNorm(n_embed)

    self.lm_head = nn.Linear(
        n_embed,
        50257
    )

  def forward(self, idx, targets=None):

    B, T = idx.shape

    tok_embed = self.token_embedding_table(idx)

    pos_embed = self.position_embedding_table(
        torch.arange(T, device=device)
    )

    x = tok_embed + pos_embed

    x = self.blocks(x)

    x = self.ln_f(x)

    logits = self.lm_head(x)

    loss = None

    if targets is not None:

      B, T, C = logits.shape

      logits = logits.view(B*T, C)

      targets = targets.view(B*T)

      loss = F.cross_entropy(
          logits,
          targets
      )

    return logits, loss

## Create Model

In [ ]:
model = GPT().to(device)

print(
    sum(
        p.numel()
        for p in model.parameters()
    ) / 1e6
    "Million Parameters"
)

## Optimizer



In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr = 1e-4,
    betas=(0.9, 0.95),
    weight_decay=0.1
)

## Training Loop

In [ ]:
for step in range(max_iters):

  xb, yb = get_batch("train")\

  logits, loss = model(xb, yb)

  optimizer.zero_grad()

  loss.backward()

  torch.nn.utils.clip_grad_norm_(
      model.parameters(),
      0.5
  )

  optimizer.step()

  if step % 100 == 0:
    print(step, loss.item())

## Save Model

In [ ]:
torch.save(
    model.state_dict(),
    "tinystories_gpt.pt"
)

## Text Generation Funtion

In [ ]:
@torch.no_grad()
def generate(
    self,
    idx,
    max_new_tokens,
    temparature = 0.1
):

   for _ in range(max_new_tokens):

      idx_cond = idx[:, -block_size:]

      logits, _ = self(idx_cond)

      logits = logits[:, -1, :]

      logits = logits / temparature

      probs = F.softmax(
          logits,
          dim=-1
      )

      idx_next = torch.multinomial(
          probs,
          num_samples=1
      )

      idx = torch.cat(
          (idx, idx_next),
          dim=1
      )

   return idx

## Generate Text

In [ ]:
prompt = "Once upon a time"

tokens = enc.encode(prompt)

context = torch.tensor(
    [tokens],
    device = device
)

generated = model.generate(
    context,
    max_new_tokens=100
)

print(
    enc.decode(
        generated[0].tolist()
    )
)